# catnip colab — stage 1 comparative evaluation

Standalone benchmark notebook for the trained Catnip Stage 1 detector (`yolo26_stage1_body_head_face.pt`) against external anime face and head detectors via `scripts/eval/benchmark_faces.py`:

- **Face:** Catnip (YOLO26n) vs `Fuyucch1/yolov8_animeface` (YOLOv8x6 @ 1280px), `zymk9/yolov5_anime` (YOLOv5s @ 640px), `deepghs/anime_face_detection` (YOLOv8n @ 640px — direct Nano peer)
- **Head:** Catnip (YOLO26n) vs `deepghs/anime_head_detection` (YOLO11n @ 640px — head peer)

Workflow: verify Cloudflare R2 (`catnip-canonical`) → clone repo, pull the sliced test dataset and trained model from R2 → run the face and head benchmarks → upload `summary.md`, `metrics.json`, and the `visuals/*.jpg` comparison panels to `s3://catnip-canonical/results/benchmarks/{face,head}/`.

Stage 1 training is kept separate in `notebooks/catnipColab.ipynb`; this notebook never writes to `models/` or `runs/`.


### 0. R2 credentials + bucket check


In [ ]:
try:
    import boto3
except ImportError:
    !pip install --quiet boto3
    import boto3

from google.colab import userdata

# R2 S3 API credentials (Cloudflare dashboard -> R2 -> API tokens).
# Add R2_ACCESS_KEY_ID / R2_SECRET_ACCESS_KEY as Colab secrets
# (the key-lock icon in the left sidebar).
R2_ENDPOINT = "https://eccb0b608bd926e793224944aba32cf4.r2.cloudflarestorage.com"
BUCKET = "catnip-canonical"

s3 = boto3.client(
    "s3",
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=userdata.get("R2_ACCESS_KEY_ID"),
    aws_secret_access_key=userdata.get("R2_SECRET_ACCESS_KEY"),
    region_name="auto",  # required by boto3, ignored by R2
)

try:
    s3.head_bucket(Bucket=BUCKET)
except Exception as err:
    raise RuntimeError(f"R2 bucket check failed for '{BUCKET}': {err}") from err

resp = s3.list_objects_v2(Bucket=BUCKET, Prefix="models/", MaxKeys=1)
print(f"R2 bucket '{BUCKET}' verified (models/ KeyCount: {resp.get('KeyCount', 0)})")


### 1. clone repo + pull dataset & trained model from R2

Clones `main` (the benchmark suite `scripts/eval/`, `src/eval/` lives there), pulls the pre-sliced dataset archive (`compressed/stage1_sliced.tar`, ~14 GB) and the trained Stage 1 deliverable `models/yolo26_stage1_body_head_face.pt` from R2.


In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/rifusaki/catnip.git"
REPO_DIR = "/content/catnip"
BRANCH = "main"

!pip install --quiet "ultralytics>=8.4.183,<9" "pydantic>=2.11,<3" "pydantic-settings>=2.11,<3" "omegaconf>=2.3,<3" "sahi>=0.11.36,<0.13" "opencv-python>=4.10,<5" "python-dotenv>=1,<2" "scikit-learn>=1.7,<2" "manga109api>=0.3.1,<0.4"

# Clone on first run; fast-forward on re-runs within the same session
if Path(REPO_DIR).exists():
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Preflight disk check (requires >= 35 GB free for download + untar)
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"Colab /content free disk: {free_gb:.1f} GB")
assert free_gb >= 35.0, f"Insufficient disk space: {free_gb:.1f} GB < 35 GB required."

LOCAL_DATA_ROOT = "/content/catnip-data-local"
os.makedirs(f"{LOCAL_DATA_ROOT}/models", exist_ok=True)
os.environ["CATNIP_DATA"] = LOCAL_DATA_ROOT

dataset_yaml = Path(f"{LOCAL_DATA_ROOT}/training/stage1_sliced/dataset.yaml")
if not dataset_yaml.exists():
    print("Downloading dataset archive from R2 (14 GB)...")
    s3.download_file(BUCKET, "compressed/stage1_sliced.tar", "/content/stage1_sliced.tar")
    print("Extracting dataset...")
    !tar -xf /content/stage1_sliced.tar -C {LOCAL_DATA_ROOT}
    !rm /content/stage1_sliced.tar

model_path = Path(f"{LOCAL_DATA_ROOT}/models/yolo26_stage1_body_head_face.pt")
if not model_path.exists():
    print("Downloading trained model yolo26_stage1_body_head_face.pt from R2...")
    s3.download_file(BUCKET, "models/yolo26_stage1_body_head_face.pt", str(model_path))

# Patch dataset.yaml to the absolute Colab path (idempotent on re-runs)
import yaml as _yaml

_data = _yaml.safe_load(dataset_yaml.read_text())
_data["path"] = str(dataset_yaml.parent.resolve())
dataset_yaml.write_text(_yaml.dump(_data, default_flow_style=False))
print(f"dataset.yaml path -> {_data['path']}")

assert dataset_yaml.exists(), f"Extraction error: {dataset_yaml} does not exist!"
assert model_path.exists(), f"Model fetch error: {model_path} does not exist!"
catnip_model = model_path  # consumed by the benchmark cells
print("Dataset + trained model verified on local NVMe!")
!nvidia-smi -L || true
print(f"CATNIP_DATA={os.environ.get('CATNIP_DATA', 'UNSET')}")


### 2. face detection benchmark — Catnip vs Fuyucchi, zymk9, DeepGHS Nano

External benchmark weights (Fuyucchi YOLOv8x6, zymk9 YOLOv5s, DeepGHS Face Nano) are downloaded from HuggingFace/GitHub on first run and cached in `~/.cache/catnip/benchmarks/`.

`SAMPLE = 500` keeps a run to a few minutes on an L4; set `0` for the full ~16k test split.


In [ ]:
import sys, shutil, subprocess
from pathlib import Path
from IPython.display import Markdown, display
from src.config import settings

assert Path("scripts/eval/benchmark_faces.py").exists(), "Run the setup cell first (repo must be the working directory)."

print(f"Using Catnip model: {catnip_model}")

SAMPLE = 500  # 0 = full test split
DEVICE = "cuda"

face_output_dir = settings.paths.output_dir / "benchmarks_face"

# Fresh output dir per run: keeps every artifact tied to the current run only
shutil.rmtree(face_output_dir, ignore_errors=True)

cmd = [
    sys.executable, "scripts/eval/benchmark_faces.py",
    "--target", "face",
    "--models", "catnip,fuyucchi_face,zymk9_face,deepghs_face_nano",
    "--catnip-model", str(catnip_model),
    "--dataset-yaml", str(Path(f"{LOCAL_DATA_ROOT}/training/stage1_sliced/dataset.yaml")),
    "--device", DEVICE,
    "--sample", str(SAMPLE),
    "--output-dir", str(face_output_dir),
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

summary_path = face_output_dir / "summary.md"
if summary_path.exists():
    display(Markdown(summary_path.read_text()))


### 3. head detection benchmark — Catnip vs DeepGHS Head Nano


In [ ]:
import sys, shutil, subprocess
from pathlib import Path
from IPython.display import Markdown, display
from src.config import settings

assert Path("scripts/eval/benchmark_faces.py").exists(), "Run the setup cell first (repo must be the working directory)."

print(f"Using Catnip model: {catnip_model}")

SAMPLE = 500  # 0 = full test split
DEVICE = "cuda"

head_output_dir = settings.paths.output_dir / "benchmarks_head"

# Fresh output dir per run: keeps every artifact tied to the current run only
shutil.rmtree(head_output_dir, ignore_errors=True)

cmd_head = [
    sys.executable, "scripts/eval/benchmark_faces.py",
    "--target", "head",
    "--models", "catnip,deepghs_head_nano",
    "--catnip-model", str(catnip_model),
    "--dataset-yaml", str(Path(f"{LOCAL_DATA_ROOT}/training/stage1_sliced/dataset.yaml")),
    "--device", DEVICE,
    "--sample", str(SAMPLE),
    "--output-dir", str(head_output_dir),
]
print("Running:", " ".join(cmd_head))
subprocess.run(cmd_head, check=True)

summary_head_path = head_output_dir / "summary.md"
if summary_head_path.exists():
    display(Markdown(summary_head_path.read_text()))


### 4. upload results to R2

Uploads `summary.md`, `metrics.json`, and every panel in `visuals/*.jpg` for both targets to `s3://catnip-canonical/results/benchmarks/{face,head}/`.


In [ ]:
from pathlib import Path
from src.config import settings

def list_remote_keys(prefix: str) -> list:
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        keys.extend(obj["Key"] for obj in page.get("Contents", []))
    return keys

def upload_eval_results(target: str, local_dir) -> None:
    """Upload the latest run's summary.md, metrics.json, and visuals/*.jpg to results/benchmarks/<target>/."""
    local_dir = Path(local_dir)
    if not local_dir.exists():
        print(f"[{target}] no results at {local_dir} (benchmark cell not run?) — skipping")
        return
    prefix = f"results/benchmarks/{target}"
    reports = 0
    for filename in ("summary.md", "metrics.json"):
        report = local_dir / filename
        if report.exists():
            s3.upload_file(str(report), BUCKET, f"{prefix}/{filename}")
            reports += 1
        else:
            print(f"[{target}] missing {filename} — benchmark may have failed")
    visuals_dir = local_dir / "visuals"
    panels = sorted(visuals_dir.glob("*.jpg")) if visuals_dir.is_dir() else []
    stale_panels = []
    if panels:
        stale_panels = list_remote_keys(f"{prefix}/visuals/")
        for i in range(0, len(stale_panels), 1000):
            s3.delete_objects(
                Bucket=BUCKET,
                Delete={"Objects": [{"Key": k} for k in stale_panels[i:i + 1000]], "Quiet": True},
            )
    for panel in panels:
        s3.upload_file(str(panel), BUCKET, f"{prefix}/visuals/{panel.name}")
    print(f"[{target}] uploaded {reports} reports + {len(panels)} visual panels (pruned {len(stale_panels)} stale) to s3://{BUCKET}/{prefix}/")

upload_eval_results("face", settings.paths.output_dir / "benchmarks_face")
upload_eval_results("head", settings.paths.output_dir / "benchmarks_head")
print("Done! Compare results at s3://catnip-canonical/results/benchmarks/")
